# What you will produce today
1. A **Grad-CAM heatmap** you compute yourself in about 10 lines of PyTorch, showing which part of a photo made the model choose its answer.
2. A **check** that your heatmap matches the `pytorch-grad-cam` library (the tool you'd use in a real project).
3. A **sanity test** that proves the heatmap means something: cover the highlighted area and watch the model's confidence collapse.

# How to use this notebook
* Run the cells **from top to bottom** (`Shift + Enter`).
* Compare your output to what's written. If it doesn't match, raise your hand. A helper will come to you.
* Stop and think (or talk with your neighbor) before running more cells.

# The project need

Your team is building a photo auto-tagger for a website: users upload photos, and a pretrained image classifier suggests tags like *"espresso"* or *"cat"*. You didn't train the model. You downloaded a well-known one.

Accuracy on a test set looks fine. But before shipping, your manager asks:

> *"When the model says 'espresso', is it actually looking at the coffee, or at something else in the picture, like the table or a logo?"*

A model can be right for the wrong reasons. A famous example is a "wolf vs. husky" classifier that was really detecting **snow in the background**. It worked on the test set and failed in the real world.
**Grad-CAM** lets you *see* which part of the image the model used for its answer.

# The key idea in 60 seconds

A CNN turns an image into many **feature maps**: small grids where each cell says "I see pattern *k* here." In the last convolutional layer of our model there are **512 feature maps, each 7 × 7**. Each cell covers about a 32 × 32-pixel patch of the photo.

Grad-CAM asks two questions:
1. **Which feature maps matter for this class?** Use the **gradient**: if nudging a feature map up would raise the "espresso" score a lot, that map matters.
2. **Where are those important feature maps active?** Add up the feature maps, each weighted by how much it matters, and keep only the positive part.

The result is a 7 × 7 "importance grid" that we stretch to the photo's size and draw as a heatmap. **Red = this region pushed the model toward the class.**

| Step | What happens | Shape (our model) |
|---|---|---|
| 1. Forward pass | run the image, **save** the last conv layer's feature maps and the class scores | maps `(1, 512, 7, 7)`, scores `(1, 1000)` |
| 2. Pick a class | take that class's **raw score** (before softmax) | a single number |
| 3. Gradients | how the score changes with each feature-map cell | `(1, 512, 7, 7)` |
| 4. Combine | average each gradient map → 1 importance number per map; weighted sum of maps; ReLU | `(7, 7)` |
| 5. Show it | stretch to `224 × 224`, overlay on the photo | `(224, 224)` |

---
# Step 0: Setup

**In Google Colab:** PyTorch, torchvision and scikit-image are already installed. Run the cell below to add the Grad-CAM library (≈20 s). **No GPU needed.** The default CPU runtime is fine.
**On your own computer:** run `pip install -r requirements.txt` in the workshop folder first

There is **no data to download**: the photos come with scikit-image, and the model weights download automatically in Step 2.

In [ ]:
%pip install -q grad-cam==1.5.7

# Step 1: Imports

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from skimage import data, transform               
from torchvision.models import resnet18, ResNet18_Weights

print("PyTorch:", torch.__version__)

**CHECK:** If a PyTorch version prints (2.x), MOVE FOWARD

## Step 2: Load a pretrained model


In [ ]:
pretrained = ResNet18_Weights.DEFAULT                 # these are the pretrained weights from RESNET
model = resnet18(weights=pretrained).eval()           
categories = pretrained.meta["categories"]            

print("Number of classes:", len(categories))
print("Class 967 is:", categories[967])
print("Parameters:", sum(p.numel() for p in model.parameters()))

**CHECK:** `Number of classes: 1000`, `Class 967 is: espresso`, about 11.7 million parameters

## Step 3: Load a photo and prepare it for the model

The model expects images prepared **exactly** the way its training images were: resized, center-cropped to **224 × 224**, and normalized color values. `pretrained.transforms()` does all of that for us.

Two small helpers:
* `preprocess(photo)`: NumPy photo → model input of shape `(1, 3, 224, 224)` (1 image, 3 color channels, 224 × 224 pixels).
* `show(x)`: undo the normalization so we can **see exactly what the model sees** (including the crop).

In [ ]:
model_prep = pretrained.transforms()
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def preprocess(photo):
    """NumPy image (height, width, 3) -> model input (1, 3, 224, 224)."""
    return model_prep(Image.fromarray(photo)).unsqueeze(0)    # unsqueeze adds the 'batch' dimension

def show(x):
    """Model input -> a normal picture (224, 224, 3) we can plot."""
    return (x[0] * STD + MEAN).clamp(0, 1).permute(1, 2, 0).numpy()

photo = data.coffee()
x = preprocess(photo)
print("photo:", photo.shape, "-> model input:", tuple(x.shape))

plt.figure(figsize=(3, 3)); plt.imshow(show(x)); plt.axis("off"); plt.title("what the model sees"); plt.show()

## Step 4: What does the model predict?

`torch.no_grad()` = "just predict, don't keep track of gradients". That's faster, and we don't need gradients here (we will in Step 5).
`softmax` turns the 1,000 raw scores into probabilities that add up to 1.

In [ ]:
def top5(x):
    with torch.no_grad():
        probs = model(x).softmax(dim=1)[0]
    values, indices = probs.topk(5)
    for p, i in zip(values, indices):
        print(f"  {p:6.1%}  {categories[i]}  (class {i})")

top5(x)

# this prints the models 5 likely classes for one image

**CHECK:** the top answer should be **espresso, about 85%**. Small differences in the decimals between PyTorch versions are normal.

This is good, but **why** did it say espresso? Let's build Grad-CAM, one step at a time.

## Step 5: Grad-CAM step 1. Forward pass and save the feature maps

We need the output of the **last convolutional block** (`model.layer4`), but the model only returns the final scores.
A **hook** solves this: it's a small function PyTorch calls for you during the forward pass, so you can grab a layer's output on the way through.

In [ ]:
saved = {}

def save_output(module, inputs, output):
    saved["maps"] = output                           # remember what layer4 produced

hook = model.layer4.register_forward_hook(save_output)
scores = model(x)                                    # forward pass, this time WITH gradients
hook.remove()                                        # clean up so the hook doesn't stay attached

feature_maps = saved["maps"]
print("feature maps:", tuple(feature_maps.shape))
print("scores:      ", tuple(scores.shape))

**CHECK:** `feature maps: (1, 512, 7, 7)` and `scores: (1, 1000)`.
That's 512 feature maps, each a 7 × 7 grid over the photo, plus one raw score per class.

## Step 6: Grad-CAM steps 2–3. Pick a class and take the gradient


We take the **raw score** for "espresso", *before* softmax. (After softmax, the number also depends on all the other 999 classes, which muddies the answer.)

Then `torch.autograd.grad` answers: *for every cell of every feature map, how much would the espresso score change if that cell went up a little?* That's the same backpropagation you use in training, but we stop at `layer4` instead of going all the way back to the weights.

In [ ]:
target = categories.index("espresso")               # 967
espresso_score = scores[0, target]                  # one number
grads = torch.autograd.grad(espresso_score, feature_maps)[0]

print("espresso raw score:", round(espresso_score.item(), 2))
print("gradients:", tuple(grads.shape))

**CHECK:** `gradients: (1, 512, 7, 7)`, **the same shape as the feature maps**. One gradient value per feature-map cell.